# Optional GPU Benchmark

This notebook is an **opt-in experiment**. The toolkit is CPU-only by default,
and every cell below is written to degrade gracefully when no GPU, no NVIDIA
driver, or no `pynvml` is available.

## What this notebook checks

1. Is `pynvml` importable?
2. Is the NVIDIA driver visible to the container?
3. Does the `GpuMonitor` report a usable device?
4. Can `torch` see a CUDA device?
5. A small, deterministic CPU vs. GPU embedding benchmark on a synthetic batch.

If any check fails, the rest of the notebook reports the reason and exits the
benchmark section cleanly — it does not raise.

## How to run with GPU

From the project root:

```bash
docker compose -f docker-compose.yml -f docker-compose.gpu.yml up --build
```

This requires an NVIDIA GPU on the host and `nvidia-container-toolkit`
installed.

## 1. pynvml availability

In [ ]:
try:
    import pynvml
    PYNVML_AVAILABLE = True
    print('pynvml: importable')
except ImportError as exc:
    PYNVML_AVAILABLE = False
    print(f'pynvml: NOT available ({exc})')
    print('This notebook will only run the CPU half of the benchmark.')

## 2. NVIDIA driver visible to the container

In [ ]:
import subprocess

try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    NVIDIA_SMI_OK = out.returncode == 0
    if NVIDIA_SMI_OK:
        print(out.stdout.strip())
    else:
        print(f'nvidia-smi exit {out.returncode}: {out.stderr.strip()}')
except FileNotFoundError:
    NVIDIA_SMI_OK = False
    print('nvidia-smi: not installed in this container.')
except Exception as exc:
    NVIDIA_SMI_OK = False
    print(f'nvidia-smi failed: {exc}')

## 3. GpuMonitor report

Uses the toolkit's own monitoring class. It is safe to call on CPU-only
machines: it returns a `GpuInfo` with `available=False` and `error` set to a
human-readable reason.

In [ ]:
from rag.evaluation.monitors.gpu_monitor import GpuMonitor

monitor = GpuMonitor(gpu_index=0)
info = monitor.info()
print(info.to_dict())

GPU_AVAILABLE = info.available

## 4. torch.cuda visibility

In [ ]:
try:
    import torch
    TORCH_CUDA_OK = bool(torch.cuda.is_available())
    print(f'torch.__version__       = {torch.__version__}')
    print(f'torch.cuda.is_available = {TORCH_CUDA_OK}')
    if TORCH_CUDA_OK:
        print(f'torch.cuda.device_count = {torch.cuda.device_count()}')
        print(f'current device name     = {torch.cuda.get_device_name(0)}')
except ImportError as exc:
    TORCH_CUDA_OK = False
    print(f'torch not importable: {exc}')

## 5. Optional CPU vs. GPU embedding benchmark

Synthetic deterministic input (no network, no HF download required for the
warm path — only if the model isn't cached). The benchmark times a fixed
number of `model.encode` passes on a CPU device and, if a GPU is reachable,
repeats the same passes on CUDA.

**Skip condition:** if no GPU is available, only the CPU baseline runs.

In [ ]:
import time
from typing import List

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 32
N_BATCHES = 4
SEQ_LEN_TARGET = 64  # rough — sentence-transformers handles its own tokenisation.

def make_batch(n: int) -> List[str]:
    base = 'the quick brown fox jumps over the lazy dog ' * 4
    return [f'{i} {base}' for i in range(n)]

BATCHES = [make_batch(BATCH_SIZE) for _ in range(N_BATCHES)]
print(f'Prepared {N_BATCHES} batches of {BATCH_SIZE} synthetic sentences.')

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
except ImportError as exc:
    ST_AVAILABLE = False
    print(f'sentence-transformers not installed: {exc}')

In [ ]:
def time_encode(device: str) -> float:
    '''Return total wall-clock seconds across all batches.'''
    model = SentenceTransformer(MODEL_NAME, device=device)
    # Warmup — one batch outside the measurement window.
    model.encode(BATCHES[0], convert_to_numpy=True)
    t0 = time.perf_counter()
    for batch in BATCHES:
        model.encode(batch, convert_to_numpy=True)
    return time.perf_counter() - t0

results = {}
if ST_AVAILABLE:
    cpu_seconds = time_encode('cpu')
    results['cpu'] = cpu_seconds
    print(f'CPU: {cpu_seconds:.3f} s for {N_BATCHES} batches of {BATCH_SIZE}')
else:
    print('Skipping CPU benchmark (sentence-transformers not installed).')

In [ ]:
if ST_AVAILABLE and TORCH_CUDA_OK and GPU_AVAILABLE:
    gpu_seconds = time_encode('cuda')
    results['cuda'] = gpu_seconds
    print(f'GPU: {gpu_seconds:.3f} s for {N_BATCHES} batches of {BATCH_SIZE}')
    if 'cpu' in results:
        speedup = results['cpu'] / gpu_seconds if gpu_seconds > 0 else float('inf')
        print(f'Speedup CPU/GPU = {speedup:.2f}x')
else:
    reasons = []
    if not ST_AVAILABLE:
        reasons.append('sentence-transformers missing')
    if not TORCH_CUDA_OK:
        reasons.append('torch.cuda.is_available() is False')
    if not GPU_AVAILABLE:
        reasons.append('GpuMonitor reports no GPU')
    print('Skipping GPU benchmark: ' + '; '.join(reasons))
    print('This is expected on CPU-only deployments — not an error.')

## 6. Resource snapshot example

Demonstrates that `ResourceSnapshotCollector` with `gpu_monitoring=True` is
safe even on CPU-only machines — the GPU fields are simply `None`.

In [ ]:
from rag.evaluation.monitors.resource_snapshot import ResourceSnapshotCollector

collector = ResourceSnapshotCollector(gpu_monitoring=True)
snap = collector.collect()
for field, value in snap.to_dict().items():
    print(f'{field:>28}: {value}')